<a href="https://colab.research.google.com/github/RPGNZ/Rebalancing-engine-with-flat-tax/blob/main/portfolio_rebalancing_flat_tax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Portfolio rebalancing engine with French Flat Tax (aka PFU - Prélèvement Forfaitaire Unique)

This notebook demonstrates a realistic portfolio rebalancing engine with:

- Multi-asset portfolio allocation
- Monthly DCA contributions
- Rebalancing toward target weights
- French flat-tax (PFU = 31.4%) applied on realized gains
- Iterative convergence taking into account taxes reducing available capital

The objective is to validate the consistency and robustness of the rebalancing logic through several simple and interpretable test cases.


## Imports

In [21]:
import pandas as pd
import numpy as np

## Rebalancing Engine

This implementation:

1. Computes the portfolio value
2. Adds monthly DCA contribution
3. Computes the target allocation
4. Detects required sales
5. Applies taxes on realized gains
6. Recomputes the target allocation after taxes
7. Iterates until convergence


In [22]:
def rebalance_with_tax_iterative(
    prices_t,
    units,
    weights,
    cost_basis,
    monthly_DCA,
    tax_rate=0.314,
    max_iter=20,
    tol=1e-6
):

    weights = pd.Series(weights, dtype=float)
    weights = weights / weights.sum()

    prices_t = pd.Series(prices_t)[weights.index]
    units = pd.Series(units)[weights.index]
    cost_basis = pd.Series(cost_basis)[weights.index]

    gross_value = (units * prices_t).sum() + monthly_DCA

    current_value = gross_value
    prev_units_after = None

    for i in range(max_iter):

        # Target allocation
        units_target = (weights * current_value) / prices_t

        # Positive = buy / Negative = sell
        units_delta = units_target - units

        # Only sales generate taxes
        units_sold = (-units_delta).clip(lower=0)

        # Realized gains
        pnl_realized = (prices_t - cost_basis) * units_sold

        # Tax only on positive gains
        tax = pnl_realized.clip(lower=0) * tax_rate

        total_tax = tax.sum()

        # Net capital after tax
        net_value = gross_value - total_tax

        # Recompute target allocation
        units_after = (weights * net_value) / prices_t

        # Convergence check
        if prev_units_after is not None:
            diff = (units_after - prev_units_after).abs().sum()

            if diff < tol:
                break

        prev_units_after = units_after.copy()
        current_value = net_value

    # Final metrics
    final_value = (units_after * prices_t).sum()

    summary = pd.DataFrame({
        "Price": prices_t,
        "Cost Basis": cost_basis,
        "Units Before": units,
        "Units After": units_after,
        "Units Delta": units_after - units,
        "Realized PnL": pnl_realized,
        "Tax Paid": tax
    })

    return units_after, summary, final_value, total_tax, i + 1

# Use Case 1 — Simple Two-Asset Rebalance

Expected behavior:

- Asset A strongly outperformed
- Rebalancing should trigger sales on Asset A
- Taxes should reduce the final portfolio value


In [31]:
weights = {
    "Asset_A": 0.5,
    "Asset_B": 0.5
}

prices_t = pd.Series({
    "Asset_A": 150,
    "Asset_B": 200
})

units = pd.Series({
    "Asset_A": 10,
    "Asset_B": 5
})

cost_basis = pd.Series({
    "Asset_A": 100,
    "Asset_B": 200
})

units_after, summary, final_value, total_tax, n_iter = rebalance_with_tax_iterative(
    prices_t=prices_t,
    units=units,
    weights=weights,
    cost_basis=cost_basis,
    monthly_DCA=0
)

print(summary.round(4))
print(f"\nFinal portfolio value : {final_value:.2f}")
print(f"Total tax paid        : {total_tax:.2f}")
print(f"Iterations            : {n_iter}")


         Price  Cost Basis  Units Before  Units After  Units Delta  \
Asset_A    150         100            10       8.2413      -1.7587   
Asset_B    200         200             5       6.1810       1.1810   

         Realized PnL  Tax Paid  
Asset_A       87.9353   27.6117  
Asset_B        0.0000    0.0000  

Final portfolio value : 2472.39
Total tax paid        : 27.61
Iterations            : 6


## Validation Points

- Asset_A is partially sold
- Asset_B is bought
- Taxes are strictly positive
- Final portfolio value is lower than gross value because of taxes
- Asset_A and Asset_B values shall match expected weights

In [32]:
print(f"Sell Asset_A                : {summary.loc["Asset_A", "Units Delta"]:.3f} @ {prices_t["Asset_A"]:.2f} = {summary.loc["Asset_A", "Units Delta"]*prices_t["Asset_A"]:.2f}")
print(f"Asset_A value after sell    : ({units["Asset_A"]:.2f} {summary.loc["Asset_A", "Units Delta"]:.2f}) * {prices_t["Asset_A"]:.2f} = {(units["Asset_A"] + summary.loc["Asset_A", "Units Delta"])*prices_t["Asset_A"]:.2f}")
print(f"Tax to pay                  : 31.4% * {summary.loc["Asset_A", "Units Delta"]:.2f} * {(prices_t["Asset_A"] - cost_basis["Asset_A"]):.2f} = {0.314 * summary.loc["Asset_A", "Units Delta"] * (prices_t["Asset_A"] - cost_basis["Asset_A"]):.2f}")
print(f"Portfolio value after taxes : {(units * prices_t).sum():.2f} {0.314 * summary.loc["Asset_A", "Units Delta"] * (prices_t["Asset_A"] - cost_basis["Asset_A"]):.2f} = {(units * prices_t).sum() + 0.314 * summary.loc["Asset_A", "Units Delta"] * (prices_t["Asset_A"] - cost_basis["Asset_A"]):.2f}")
print(f"Asset_A weight after sell   : {(units["Asset_A"] + summary.loc["Asset_A", "Units Delta"])*prices_t["Asset_A"]/((units * prices_t).sum() + 0.314 * summary.loc["Asset_A", "Units Delta"] * (prices_t["Asset_A"] - cost_basis["Asset_A"])):.3f}")
print(f"\nBuy Asset_B                 : {summary.loc["Asset_B", "Units Delta"]:.3f} @ {prices_t["Asset_B"]:.2f} = {summary.loc["Asset_B", "Units Delta"]*prices_t["Asset_B"]:.2f}")
print(f"Asset_B value after buy     : ({units["Asset_B"]:.2f} + {summary.loc["Asset_B", "Units Delta"]:.2f}) * {prices_t["Asset_B"]:.2f} = {(units["Asset_B"] + summary.loc["Asset_B", "Units Delta"])*prices_t["Asset_B"]:.2f}")
print(f"Asset_B weight after sell   : {(units["Asset_B"] + summary.loc["Asset_B", "Units Delta"])*prices_t["Asset_B"]/((units * prices_t).sum() + 0.314 * summary.loc["Asset_A", "Units Delta"] * (prices_t["Asset_A"] - cost_basis["Asset_A"])):.3f}")

Sell Asset_A                : -1.759 @ 150.00 = -263.81
Asset_A value after sell    : (10.00 -1.76) * 150.00 = 1236.19
Tax to pay                  : 31.4% * -1.76 * 50.00 = -27.61
Portfolio value after taxes : 2500.00 -27.61 = 2472.39
Asset_A weight after sell   : 0.500

Buy Asset_B                 : 1.181 @ 200.00 = 236.19
Asset_B value after buy     : (5.00 + 1.18) * 200.00 = 1236.19
Asset_B weight after sell   : 0.500


# Use Case 2 — No Taxes Scenario

Expected behavior:

- Assets have realized gains while reaching target weights
- Rebalancing should generate zero taxes


In [25]:
weights = {
    "Asset_A": 0.5,
    "Asset_B": 0.5
}

prices_t = pd.Series({
    "Asset_A": 150,
    "Asset_B": 300
})

units = pd.Series({
    "Asset_A": 8,
    "Asset_B": 4
})

cost_basis = pd.Series({
    "Asset_A": 100,
    "Asset_B": 100
})

units_after, summary, final_value, total_tax, n_iter = rebalance_with_tax_iterative(
    prices_t=prices_t,
    units=units,
    weights=weights,
    cost_basis=cost_basis,
    monthly_DCA=0
)

print(summary.round(4))
print(f"\nFinal portfolio value : {final_value:.2f}")
print(f"Total tax paid        : {total_tax:.2f}")


         Price  Cost Basis  Units Before  Units After  Units Delta  \
Asset_A    150         100             8          8.0          0.0   
Asset_B    300         100             4          4.0          0.0   

         Realized PnL  Tax Paid  
Asset_A          -0.0      -0.0  
Asset_B          -0.0      -0.0  

Final portfolio value : 2400.00
Total tax paid        : 0.00


In [26]:
weights = {
    "Asset_A": 0.5,
    "Asset_B": 0.5
}

prices_t = pd.Series({
    "Asset_A": 100,
    "Asset_B": 100
})

units = pd.Series({
    "Asset_A": 80,
    "Asset_B": 20
})

cost_basis = pd.Series({
    "Asset_A": 100,
    "Asset_B": 100
})

units_after, summary, final_value, total_tax, n_iter = rebalance_with_tax_iterative(
    prices_t=prices_t,
    units=units,
    weights=weights,
    cost_basis=cost_basis,
    monthly_DCA=0
)

print(summary.round(4))
print(f"\nFinal portfolio value : {final_value:.2f}")
print(f"Total tax paid        : {total_tax:.2f}")


         Price  Cost Basis  Units Before  Units After  Units Delta  \
Asset_A    100         100            80         50.0        -30.0   
Asset_B    100         100            20         50.0         30.0   

         Realized PnL  Tax Paid  
Asset_A           0.0       0.0  
Asset_B           0.0       0.0  

Final portfolio value : 10000.00
Total tax paid        : 0.00


## Validation Points

- Rebalancing still occurs
- Taxes remain exactly zero
- Final portfolio value remains unchanged


# Use Case 3 — DCA Injection Reduces Sales

Expected behavior:

- New capital contribution partially offsets the need to sell winners
- Taxes should be lower than without DCA


In [27]:
weights = {
    "BTC": 0.5,
    "GLD": 0.5
}

prices_t = pd.Series({
    "BTC": 100000,
    "GLD": 300
})

units = pd.Series({
    "BTC": 0.20,
    "GLD": 10
})

cost_basis = pd.Series({
    "BTC": 40000,
    "GLD": 250
})

units_after, summary, final_value, total_tax, n_iter = rebalance_with_tax_iterative(
    prices_t=prices_t,
    units=units,
    weights=weights,
    cost_basis=cost_basis,
    monthly_DCA=5000
)

print(summary.round(6))
print(f"\nFinal portfolio value : {final_value:.2f}")
print(f"Total tax paid        : {total_tax:.2f}")


      Price  Cost Basis  Units Before  Units After  Units Delta  Realized PnL  \
BTC  100000       40000           0.2     0.133760    -0.066240   3974.387257   
GLD     300         250          10.0    44.586737    34.586737      0.000000   

        Tax Paid  
BTC  1247.957599  
GLD     0.000000  

Final portfolio value : 26752.04
Total tax paid        : 1247.96


## Validation Points

- DCA cash helps rebalance toward GLD
- Taxes are reduced compared to a pure rebalance


# Use Case 4 — Convergence Stability Test

Expected behavior:

- Convergence should occur in only a few iterations
- Iterative process should remain numerically stable


In [28]:
weights = {
    "A": 0.25,
    "B": 0.25,
    "C": 0.25,
    "D": 0.25
}

prices_t = pd.Series({
    "A": 400,
    "B": 50,
    "C": 120,
    "D": 20
})

units = pd.Series({
    "A": 100,
    "B": 100,
    "C": 100,
    "D": 100
})

cost_basis = pd.Series({
    "A": 100,
    "B": 45,
    "C": 110,
    "D": 18
})

units_after, summary, final_value, total_tax, n_iter = rebalance_with_tax_iterative(
    prices_t=prices_t,
    units=units,
    weights=weights,
    cost_basis=cost_basis,
    monthly_DCA=0
)

print(summary.round(6))
print(f"\nIterations required : {n_iter}")
print(f"Total tax paid      : {total_tax:.2f}")


   Price  Cost Basis  Units Before  Units After  Units Delta  Realized PnL  \
A    400         100           100    32.926019   -67.073981   20122.19418   
B     50          45           100   263.408155   163.408155       0.00000   
C    120         110           100   109.753398     9.753398       0.00000   
D     20          18           100   658.520388   558.520388       0.00000   

      Tax Paid  
A  6318.368972  
B     0.000000  
C     0.000000  
D     0.000000  

Iterations required : 8
Total tax paid      : 6318.37


# Key Takeaways

This notebook demonstrates several important concepts:

- Taxes create a recursive dependency in portfolio rebalancing
- A naïve one-pass rebalance is generally inconsistent
- Iterative convergence is a robust and practical solution
- DCA contributions naturally reduce taxable sales
- Buy-only rebalancing can further minimize tax drag
